In [50]:
import pandas as pd
import mysql.connector
from sqlalchemy import create_engine, text
import os
from dotenv import load_dotenv
from datetime import datetime
import numpy as np
from tqdm import tqdm
import logging
from etl import ETL

In [51]:
def old_paswitch_prepare(df: pd.DataFrame):
    """Prepare dataframe by adding foreign key columns and handling nulls"""
    # Add the foreign key columns with NULL values
    df['import_id'] = None
    column_mapping = {
        'Seq': 'seq',
        'Name': 'egs',
        'Commodity': 'commodity',
        'PriceStructure': 'rate_type',
        'Rate': 'rate',
        'TermLength': 'term',
        'State': 'state',
        'Zip': 'zip',
        'RenewablePercentage': 'green_percentage',
        'EnergySource': 'green_source',
        'Service': 'service_type',
        'Date': 'date',
        'MainPUC': 'edc',
        'Web': 'egs_website',
        'Phone': 'egs_phone',
        'DiscountAvailable': 'discount_available',
        'IntroPrice': 'intro_price',
        'IsRenewable': 'is_green',
        'CancelFee': 'cancel_fee',
        'MonthlyFee': 'monthly_fee_amount',
        'NetMetering': 'net_metering',
        'EndDate': 'plan_end_date',
        'EnrollFee': 'enroll_fee',
        'RateType': 'rate_schedule'
    }
    df = df.rename(columns=column_mapping)
    df['enroll_fee'] = None
    df['has_monthly_fee'] = df['monthly_fee_amount'].apply(lambda x: 1 if pd.notnull(x) and x > 0 else 0)
    df['edc_phone'] = None
    df['edc_website'] = None
    df['unlimited_rate'] = None
    df['sign_up_link'] = None
    df['more_info'] = None
    
    # Replace NaN with None for proper NULL handling in MySQL
    df = df.where(pd.notnull(df), None)
    
    return df

def new_paswitch_prepare(df: pd.DataFrame):
    df['import_id'] = None
    df['date'] = pd.to_datetime(df['date'])

    # Let pandas infer multiple date formats (handles both "14-Dec-23" and "November 11, 2024")
    df['plan_end_date_temp'] = pd.to_datetime(df['plan_end_date'], errors='coerce')

    # Create boolean column
    df['has_plan_end_date'] = (
        df['plan_end_date_temp'].notna() |  # Has valid date (any format)
        (df['plan_end_date'].str.upper() == 'YES')  # Or is "YES"
    )

    # Set plan_end_date to the parsed date, or NULL for YES/NO
    df['plan_end_date'] = df['plan_end_date_temp']

    # Clean up temp column
    df.drop('plan_end_date_temp', axis=1, inplace=True)
    
    df = df.where(pd.notnull(df), None)
    return df

def old_ptc_paswitch_prepare(df: pd.DataFrame):
    df = df.rename(columns={'edc_id': 'switch_edc_id'})
    # df['switch_edc_id'] = pd.to_numeric(df['switch_edc_id'], errors='coerce')
    # df['switch_edc_id'] = df['switch_edc_id'].fillna(0).astype(int)
    df['update_date'] = pd.to_datetime(df['update_date'], errors='coerce')
    df = df.where(pd.notnull(df), None)
    return df

In [52]:
METADATA = [
    {"table_name": "PAPowerSwitch_Plans",
     "source": "PAPowerSwitch",
     "filepath": "../../data/v2/scraped_power.csv",
     "notes": "PAPowerSwitch historical data",
     "transform_function": new_paswitch_prepare},
    {"table_name": "ptc_PAPowerSwitch",
     "source": "PAPowerSwitch",
     "filepath": "../../data/processed/ptc_10_14_2025.csv",
     "notes": "PAPowerSwitch PTC historical data taken from october 14, 2025",
     "transform_function": old_ptc_paswitch_prepare}
]
CHUNK_SIZE = 10000  # Number of rows to process at once - adjust based on memory
BATCH_SIZE = 1000   # Number of rows to insert at once

In [53]:
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

load_dotenv()

True

In [54]:
DB_CONFIG = {
    'host': os.getenv('DB_HOST', 'localhost'),
    'port': int(os.getenv('DB_PORT', 3306)),
    'database': os.getenv('DB_NAME', 'EEdb'),
    'user': os.getenv('DB_USER', 'your_username'),
    'password': os.getenv('DB_PASSWORD', 'your_password')
}
db_handler = ETL(DB_CONFIG['user'], DB_CONFIG['password'], DB_CONFIG['host'], DB_CONFIG['port'], DB_CONFIG['database'], logger)

if db_handler.test_connection():
    print("✅ Database connection established")
else:
    print("❌ Database connection failed - check your .env file")

2025-11-21 12:04:21,807 - INFO - Database engine created for EEdb@energy-risk-data.cf8gqm02ipjk.us-east-1.rds.amazonaws.com
2025-11-21 12:04:22,556 - INFO - Database connection successful!


✅ Database connection established


In [55]:
sel = METADATA[0]

In [56]:
db_handler.import_large_csv(
    table_name=sel['table_name'],
    filepath=sel['filepath'],
    source=sel['source'],
    notes=sel['notes'],
    process_df_func=sel['transform_function'],
    chunk_size=CHUNK_SIZE,
    batch_size=BATCH_SIZE
)

2025-11-21 12:04:22,621 - INFO - Starting CSV import: ../../data/v2/scraped_power.csv -> PAPowerSwitch_Plans
2025-11-21 12:04:22,634 - INFO - File info - Columns: 30, Estimated rows: 30,117
2025-11-21 12:04:22,634 - INFO - Sample columns: ['seq', 'egs', 'commodity', 'rate_type', 'rate', 'term', 'state', 'zip', 'green_percentage', 'green_source']
2025-11-21 12:04:22,635 - INFO - File contains 30,117 rows and 30 columns
2025-11-21 12:04:22,800 - INFO - Created ImportHistory entry with ID: 43
2025-11-21 12:04:22,844 - INFO - Processing 30,117 rows in 4 chunks of 10,000 rows each
Importing rows:   0%|          | 0/30117 [00:00<?, ?rows/s]/var/folders/82/yhv7s3v966x_w8b5534xljcc0000gn/T/ipykernel_23336/780924360.py:47: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['date'] = pd.to_datetime(df['date'])
/var/folders/82/yhv7s3v966x_w8b5534xljcc0000gn/T/i

True